In [ ]:
# ============================================================
# CELL 0 — Install dependencies (safe to re-run)
# ============================================================
import subprocess, sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "groq", "openpyxl", "pandas"], check=True)
print("deps ok")

In [ ]:
# ============================================================
# CELL 0b — Mount Google Drive and verify file paths
# ============================================================
# Run this before anything else. Safe to re-run if already mounted.

from google.colab import drive
import os

if not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive")
else:
    print("Drive already mounted")

# ── set your actual folder path here ──────────────────────────
BASE = "/content/drive/MyDrive/PhD_Coursework/AV/NLU/Project/Gform_Preparation"
# ──────────────────────────────────────────────────────────────

files = [
    "GPT_EvalForm_Part_1 (Responses).xlsx",
    "GPT_EvalForm_Part_2 (Responses).xlsx",
]

all_ok = True
for f in files:
    full = os.path.join(BASE, f)
    if os.path.exists(full):
        print(f"  found : {f}")
    else:
        print(f"  MISSING: {full}")
        all_ok = False

if not all_ok:
    # List what's actually in the parent so you can spot the right path
    parent = os.path.dirname(BASE)
    # Walk up until we find a dir that exists and list its contents
    for check in [BASE, parent, os.path.dirname(parent)]:
        if os.path.isdir(check):
            print(f"\nContents of {check}:")
            for item in sorted(os.listdir(check)):
                print(f"    {item}")
            break
    raise FileNotFoundError(
        "\nFix BASE path above to match where your files actually live, then re-run this cell."
    )

print("all files found — good to go")

In [ ]:
# ============================================================
# CELL 1 — Paths and config
# ============================================================
import os, json, time, re
from datetime import datetime
import pandas as pd
from openpyxl import load_workbook

BASE     = "/content/drive/MyDrive/PhD_Coursework/AV/NLU/Project/Gform_Preparation"
OUT_DIR  = os.path.join(BASE, "LLM_Scores_GPT")
os.makedirs(OUT_DIR, exist_ok=True)

# Bump this any time scoring logic/order changes — old JSONs get wiped automatically
SCHEMA_VERSION = 2

# Input files
FILES = {
    "GPT_EvalForm_Part_1": os.path.join(BASE, "GPT_EvalForm_Part_1 (Responses).xlsx"),
    "GPT_EvalForm_Part_2": os.path.join(BASE, "GPT_EvalForm_Part_2 (Responses).xlsx"),
}

# Filling L4 slot (GPT); L1=41, L2=46, L3=51, L4=56 (0-indexed cols)
ANNOTATOR_SLOT  = "L4"
SLOT_COL        = {"L1": 41, "L2": 46, "L3": 51, "L4": 56}
SCORE_COL_START = SLOT_COL[ANNOTATOR_SLOT]

DATA_ROW_START  = 3   # 0-indexed; rows 0,1,2 are header rows

GROQ_API_KEY = "YOUR_API_KEY_HERE"
MODEL_ID     = "openai/gpt-oss-120b"
print("config ok")

In [ ]:
# ============================================================
# CELL 2 — Evaluation prompt (same content, same structure)
# ============================================================

SYSTEM_PROMPT = """You are an expert Odia (ଓଡ଼ିଆ) linguist and grammarian with deep specialization in spelling, script, grammatical, and error evaluation tasks. You are serving as an evaluator in an Odia GEC (Grammatical Error Correction) annotation study.

--- BACKGROUND (for your reference only — this work is already done) ---
Each Odia sentence in this study has been annotated for errors under exactly one of these five categories, applied in priority order:
1. Script Normalization — Unicode-level encoding errors: nukta misplacement, incorrect virama, vowel sign decomposition, ZWNJ/ZWJ issues, unintended ligatures.
2. Spelling & Typographical Errors — Correct encoding but wrong characters: short/long vowel confusion, phonetically similar consonant substitution, missing/extra chars, wrong word boundaries.
3. Grammatical Errors — Morphosyntactic issues: wrong verb tense/inflection, agreement errors, wrong case markers, word order, faulty copular constructions, missing punctuation.
4. Code-Mixing / Wrong Language — Roman-script words, non-Odia numerals, other Indic script characters, unnecessary loanwords.
5. Correct Sentence / No Errors — No errors present.

Annotation rules already applied:
- Each sentence contains at most one error; only the primary erroneous span is marked.
- Span selection covers only the minimal necessary erroneous unit.
- If no error exists: error span is empty, description states no error, corrected sentence equals the source.
- Priority rule: when a span could fit multiple categories, only the highest-priority one is assigned.
--- END BACKGROUND ---

--- YOUR TASK — EVALUATION ONLY ---
You will receive a source sentence alongside a submitted annotation response. The annotation contains five fields: Has Errors, Error Span, Annotated Category, Description, Corrected Sentence.

Your job is solely to judge how correct that submitted annotation is, using the four criteria below. Do not re-annotate the sentence. Do not second-guess what the correct answer should be independently — evaluate only what was submitted against the established annotation rules above.

C1 — ERROR EXISTS [0 or 1]
  1 : The submitted Has Errors flag correctly reflects whether an error is present in the source sentence.
  0 : Incorrect.

C2 — SPAN + DESCRIPTION [0, 1, or 2]
  2 : Exact span AND fully correct explanation (identifies what the error is, why it is wrong, and what the correct form should be).
  1 : Partial span and/or incomplete explanation.
  0 : Wrong span or hallucinated/irrelevant explanation. If no error: span must be empty and description must state no error exists.

C3 — ERROR CATEGORY [0 or 1]
  1 : The submitted category is correct (specific or parent-level acceptable). If no error exists, it must be "Correct Sentence / No Errors".
  0 : Incorrect.

C4 — CORRECTED SENTENCE [0, 1, or 2]
  2 : Fully correct, fluent, no new errors introduced.
  1 : Mostly correct with minor issues.
  0 : Incorrect, introduces new errors, or changes meaning. If no error: must match the source sentence exactly.

Also write ONE reason (10–30 words) summarising your overall judgment across all four components together.

Respond ONLY in this exact JSON format — no extra text, no markdown:
{
  "C1": <0 or 1>,
  "C2": <0, 1, or 2>,
  "C3": <0 or 1>,
  "C4": <0, 1, or 2>,
  "reason": "<10–30 word summary>"
}"""


def build_user_prompt(row):
    return f"""Source Sentence: {row['Source Sentence']}
Has Errors: {row['Has Errors']}
Error Span: {row['Error Span']}
Annotated Category: {row['Annotated Category']}
Description: {row['Description']}
Corrected Sentence: {row['Corrected Sentence']}"""

print("prompt ok")

In [ ]:
# ============================================================
# CELL 3 — Groq client + call function
# ============================================================
from groq import Groq

client = Groq(api_key=GROQ_API_KEY)

def call_groq(sentence_prompt):
    resp = client.chat.completions.create(
        model=MODEL_ID,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": sentence_prompt}
        ],
        temperature=0.0,
        max_tokens=4096,
        top_p=0.9,
        frequency_penalty=0,
        presence_penalty=0
    )
    return resp.choices[0].message.content.strip()


def parse_scores(raw_text):
    raw_text = re.sub(r"```(?:json)?", "", raw_text).strip().rstrip("`").strip()
    data = json.loads(raw_text)
    c1 = int(data["C1"])
    c2 = int(data["C2"])
    c3 = int(data["C3"])
    c4 = int(data["C4"])
    reason = str(data.get("reason", "")).strip()
    assert c1 in (0,1),     f"C1 bad: {c1}"
    assert c2 in (0,1,2),   f"C2 bad: {c2}"
    assert c3 in (0,1),     f"C3 bad: {c3}"
    assert c4 in (0,1,2),   f"C4 bad: {c4}"
    return c1, c2, c3, c4, reason

print("groq client ok")

In [ ]:
# ============================================================
# CELL 4 — Resume helpers
# ============================================================

def json_path(file_key):
    return os.path.join(OUT_DIR, f"progress_{file_key}_gpt.json")

def load_progress(file_key):
    p = json_path(file_key)
    if os.path.exists(p):
        with open(p) as f:
            data = json.load(f)
        if data.get("__version__") != SCHEMA_VERSION:
            print(f"  [progress] stale schema in {p} — wiping and starting fresh")
            os.remove(p)
            return {}
        return data
    return {}

def save_progress(file_key, progress):
    progress["__version__"] = SCHEMA_VERSION
    with open(json_path(file_key), "w") as f:
        json.dump(progress, f, ensure_ascii=False, indent=2)

print("resume helpers ok")

In [ ]:
# ============================================================
# ============================================================
# CELL 5 — Write output in Google Form response sheet style
# ============================================================
# The output mimics exactly what a human annotator produces
# when their Google Form responses are exported as a spreadsheet.
# One row = one model submission. Score values are label strings,
# not raw numbers — matching the human CSV format exactly.

C1_LABEL = {1: "1 — Correctly identified",        0: "0 — Incorrect identification"}
C2_LABEL = {2: "2 — Exact span AND fully correct explanation",
            1: "1 — Partial span and/or incomplete explanation",
            0: "0 — Wrong span or hallucinated/irrelevant explanation"}
C3_LABEL = {1: "1 — Correct category",             0: "0 — Incorrect category"}
C4_LABEL = {2: "2 — Fully correct, fluent, no new errors",
            1: "1 — Mostly correct with minor issues",
            0: "0 — Incorrect, introduces errors, or changes meaning"}

PROFILE_HEADERS = [
    "Timestamp", "Full Name", "Age", "Email ID",
    "Contact Information", "Odia Proficiency",
    "I voluntarily agree to participate in this evaluation study.",
]
TASK_HEADERS = (
    ["C1 — Is the error detection correct?",
     "C2 — Is the error span and description correct?",
     "C3 — Is the error category correct?",
     "C4 — Is the corrected sentence correct?"]
    * 50
)
ALL_HEADERS = PROFILE_HEADERS + TASK_HEADERS


def write_gform_response(src_path, out_path, progress, model_tag):
    """One flat row per model, matching Google Form export format."""
    from openpyxl import Workbook
    import pandas as pd

    df_raw = pd.read_excel(src_path, header=None)
    df = df_raw.iloc[DATA_ROW_START:].copy()
    df.columns = df_raw.iloc[2]
    df = df.reset_index(drop=True)

    wb = Workbook()
    ws = wb.active
    ws.title = "Form Responses"

    ws.append(ALL_HEADERS)

    profile_values = [
        datetime.now().strftime("%m/%d/%Y %H:%M:%S"),
        model_tag,
        "N/A",
        "N/A",
        "N/A",
        "N/A",
        "Yes, I agree",
    ]

    score_values = []
    for i in range(50):
        key = str(i)
        if key in progress:
            s = progress[key]
            score_values += [
                C1_LABEL[s["C1"]],
                C2_LABEL[s["C2"]],
                C3_LABEL[s["C3"]],
                C4_LABEL[s["C4"]],
            ]
        else:
            score_values += ["", "", "", ""]

    ws.append(profile_values + score_values)

    ws_r = wb.create_sheet("Reasons")
    ws_r.append(["Row #", "Sentence_ID", "Source Sentence", f"Reason ({model_tag})"])
    for i in range(50):
        key = str(i)
        if key not in progress:
            continue
        sent_id  = df.iloc[i].get("Sentence_ID", f"row{i+1}")
        src_sent = df.iloc[i].get("Source Sentence", "")
        ws_r.append([i + 1, sent_id, src_sent, progress[key].get("reason", "")])

    wb.save(out_path)
    print(f"  saved -> {out_path}")

In [ ]:
# ============================================================
# CELL 6 — Run this cell to evaluate
# TEST_MODE=True  → first 10 rows only (safe to test with)
# TEST_MODE=False → all 50 rows (full run)
# Re-running always skips already-scored rows automatically.
# ============================================================
# Set TEST_MODE=False to evaluate all 50 rows.

TEST_MODE  = True
TEST_LIMIT = 50

def evaluate_file(file_key, src_path, test_mode=True):
    print(f"\n{'='*60}")
    print(f"FILE: {file_key}  |  slot: {ANNOTATOR_SLOT}  |  test={test_mode}")
    print(f"{'='*60}")

    df_raw = pd.read_excel(src_path, header=None)
    df     = df_raw.iloc[DATA_ROW_START:].copy()
    df.columns = df_raw.iloc[2]
    df = df.reset_index(drop=True)

    visible_cols = ["Source Sentence", "Has Errors", "Error Span",
                    "Annotated Category", "Description", "Corrected Sentence"]
    df_visible = df[visible_cols].copy()

    limit    = TEST_LIMIT if test_mode else len(df_visible)
    progress = load_progress(file_key)
    out_path = os.path.join(OUT_DIR, f"{file_key}_GPT_GForm_Response.xlsx")

    for i in range(limit):
        key = str(i)
        if key in progress:
            print(f"  row {i+1:02d} — already done, skip")
            continue

        row     = df_visible.iloc[i]
        sent_id = df.iloc[i].get("Sentence_ID", f"row{i+1}")
        print(f"  row {i+1:02d} | {sent_id} | calling GPT ...", end=" ", flush=True)

        try:
            prompt = build_user_prompt(row)
            raw    = call_groq(prompt)
            c1, c2, c3, c4, reason = parse_scores(raw)
            progress[key] = {"C1": c1, "C2": c2, "C3": c3, "C4": c4, "reason": reason}
            save_progress(file_key, progress)
            print(f"C1={c1} C2={c2} C3={c3} C4={c4} | {reason[:60]}")
        except Exception as e:
            print(f"FAILED — {e}")

        time.sleep(0.3)

    write_gform_response(src_path, out_path, progress, model_tag=f"GPT_{file_key}")
    print(f"\nDone. {len(progress)} rows scored.")


# Run
for fkey, fpath in FILES.items():
    evaluate_file(fkey, fpath, test_mode=TEST_MODE)